# Myllia: Direction Notebook (Bilinear Conditional Factorization)

This notebook implements a medium-large architecture shift:

- Learn a low-rank interaction between perturbed gene embeddings and output gene embeddings
- Predict the full delta vector as a bilinear form with rank `R`
- Train with a metric-aligned weighted L1 proxy and evaluate with the official `myllia_score`

Core model:
\[
\hat D_{i,j} = \langle W_p z_{g_i},\; W_o u_j \rangle + b_j + b_i
\]
where:
- `z_{g_i}` is an embedding for the perturbed gene `g_i`
- `u_j` is an embedding for output gene `j`
- `R` is a small rank (16 to 64)

This uses `training_cells.h5ad` to build output gene embeddings.


In [1]:
# # Hybrid model: Base SVD + STRING (seq/net H5) + STRING links (protein.links)
# Uses CFG_B for links. One CV pass for epoch/alpha, then refit + submission.

# %%
import numpy as np
import pandas as pd
from pathlib import Path
import re
import json

import anndata as ad
import scipy.sparse as sp
from scipy import sparse
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold
from sklearn.preprocessing import normalize as sk_normalize

import torch
import torch.nn as nn

from myllia_metric import myllia_score

# %%
# -----------------------
# Global config
# -----------------------
SEED = 6
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(DEVICE)

# base embedding dims
EMB_DIM_PERT = 128
EMB_DIM_OUT  = 128

RANK_R = 32
DROPOUT = 0.10

LR = 2e-3 * 0.6
WD = 1e-4

EPOCHS = 400
BATCH_GENES = 16
EVAL_EVERY = 5
PATIENCE = 12
GRAD_CLIP = 1.0

# shrink alpha sweep (single CV pass)
ALPHA_GRID = np.linspace(0.0, 0.9, 46).astype(np.float32)

# metric gate params
GATE_A = 0.0
GATE_B = 0.2
EPS = 1e-12

# refit seeds
MODEL_SEEDS = [6, 7, 8]

ROOT = Path(".")

# h5ad used only for missing perts (base embedding fallback)
H5AD_PATH = ROOT / "data" / "training_cells.h5ad"

# -----------------------
# STRING links best config you provided
# -----------------------
CFG_B = dict(
    key="B_rank1_eg2_b2=1_b3=0.5",
    score_min=400,
    topk=200,
    edge_gamma=2.0,
    beta2=1.0,
    beta3=0.5,
)

CACHE_LINKS = ROOT / "cache" / "string_links_v12"
CACHE_LINKS.mkdir(parents=True, exist_ok=True)

# %%
def score_delta(dt, dp):
    dt = dt.astype(np.float32, copy=False)
    dp = dp.astype(np.float32, copy=False)
    r = myllia_score(dt, dp)
    return {
        "score": float(r.score),
        "wcos": float(r.wcos),
        "mean_term": float(r.mean_term),
        "pred_wmae": float(r.pred_wmae),
    }

def apply_shrink(pred, baseline_bg, alpha):
    a = float(alpha)
    return a * pred + (1.0 - a) * baseline_bg

# %%
# -----------------------
# Load CSVs
# -----------------------
means_path = ROOT / "data" / "training_data_means.csv"
valmap_path = ROOT / "data" / "pert_ids_val.csv"
sample_sub_path = ROOT / "data" / "sample_submission.csv"
gt_path = ROOT / "data" / "training_data_ground_truth_table.csv"

df_means = pd.read_csv(means_path)
df_valmap = pd.read_csv(valmap_path)
df_sub = pd.read_csv(sample_sub_path)
gt_df = pd.read_csv(gt_path)

gene_columns = [c for c in df_means.columns if c != "pert_symbol"]

baseline_mask = df_means["pert_symbol"].astype(str) == "non-targeting"
x_base = df_means.loc[baseline_mask, gene_columns].iloc[0].to_numpy(np.float32)

df_train = df_means.loc[~baseline_mask].reset_index(drop=True)
train_genes = df_train["pert_symbol"].astype(str).to_numpy()

X_train_means = df_train[gene_columns].to_numpy(np.float32)
D_train = X_train_means - x_base[None, :]  # (80, G)

# pert_id -> gene symbol for first 60 leaderboard rows
val_map = dict(zip(df_valmap["pert_id"].astype(str), df_valmap["pert"].astype(str)))

id_col = "pert_id" if "pert_id" in df_sub.columns else df_sub.columns[0]
sub_ids = df_sub[id_col].astype(str).tolist()

def pert_symbol_from_id(pid: str) -> str:
    return val_map.get(str(pid), str(pid))

sub_perts = [pert_symbol_from_id(pid) for pid in sub_ids]
val_targets = df_valmap["pert"].astype(str).tolist()

print("Train perts:", len(train_genes), "G:", len(gene_columns))
print("Sample submission rows:", len(df_sub))
print("Val mapping entries:", len(val_map))

Train perts: 80 G: 5127
Sample submission rows: 120
Val mapping entries: 60


In [2]:
def load_baseline_wmae_for_train_perts(gt_df: pd.DataFrame, train_genes: np.ndarray) -> np.ndarray:
    # prefer a column that matches gene symbols
    key_candidates = ["pert", "pert_symbol", "gene", "target_gene", "sgrna_symbol"]
    key_col = None
    for c in key_candidates:
        if c in gt_df.columns:
            key_col = c
            break
    if key_col is None:
        # fallback: if file already ordered per pert (rare), try using as-is
        # but keep it safe
        if "baseline_wmae" in gt_df.columns and len(gt_df["baseline_wmae"]) >= len(train_genes):
            bw = gt_df["baseline_wmae"].to_numpy(np.float32)[:len(train_genes)]
            return bw
        raise ValueError("Could not find a gene-symbol column in ground truth table for baseline_wmae alignment.")

    bw_map = gt_df.groupby(gt_df[key_col].astype(str).str.upper())["baseline_wmae"].mean()
    bw = []
    for g in train_genes:
        v = bw_map.get(str(g).upper(), np.nan)
        bw.append(v)
    bw = np.asarray(bw, dtype=np.float32)

    if np.any(~np.isfinite(bw)):
        med = np.nanmedian(bw)
        bw = np.where(np.isfinite(bw), bw, med).astype(np.float32)
        print("[warn] baseline_wmae had missing values; filled with median:", float(med))

    return bw

baseline_wmae = load_baseline_wmae_for_train_perts(gt_df, train_genes)
baseline_wmae_t = torch.tensor(baseline_wmae, device=device, dtype=torch.float32)

In [3]:
X = D_train.astype(np.float32, copy=True)
X = np.sign(X) * np.log2(1.0 + np.abs(X))

svd = TruncatedSVD(n_components=max(EMB_DIM_PERT, EMB_DIM_OUT), random_state=SEED)
svd.fit(X)

gene_emb_all = svd.components_.T.astype(np.float32)  # (G, k)
print("SVD k:", gene_emb_all.shape[1], "gene_emb_all:", gene_emb_all.shape)

gene2emb_pert = {gene_columns[i].upper(): gene_emb_all[i, :EMB_DIM_PERT].copy() for i in range(len(gene_columns))}
gene2emb_out  = {gene_columns[i].upper(): gene_emb_all[i, :EMB_DIM_OUT ].copy() for i in range(len(gene_columns))}

emb_fallback_pert = gene_emb_all[:, :EMB_DIM_PERT].mean(axis=0).astype(np.float32)
emb_fallback_out  = gene_emb_all[:, :EMB_DIM_OUT ].mean(axis=0).astype(np.float32)

missing_emb_pert = {}

def emb_pert(g: str) -> np.ndarray:
    gU = str(g).upper()
    if gU in gene2emb_pert:
        return gene2emb_pert[gU]
    if gU in missing_emb_pert:
        return missing_emb_pert[gU]
    return emb_fallback_pert

def emb_out(g: str) -> np.ndarray:
    return gene2emb_out.get(str(g).upper(), emb_fallback_out)

U_out_base = np.vstack([emb_out(g) for g in gene_columns]).astype(np.float32)    # (G, d_out)
Z_train_base = np.vstack([emb_pert(g) for g in train_genes]).astype(np.float32)  # (N, d_pert)

print("U_out_base:", U_out_base.shape, "Z_train_base:", Z_train_base.shape)


SVD k: 80 gene_emb_all: (5127, 80)
U_out_base: (5127, 80) Z_train_base: (80, 80)


In [4]:
geneU = pd.Index([str(g).upper() for g in gene_columns])

missing_train = [g for g in train_genes.tolist() if str(g).upper() not in geneU]
missing_val = [g for g in val_targets if str(g).upper() not in geneU]
missing_sub = [g for g in sub_perts if str(g).upper() not in geneU]

missing_all = sorted(set([str(x).upper() for x in (missing_train + missing_val + missing_sub)]))
if missing_train:
    print(f"train perts not in gene_columns: {len(missing_train)}. Example: {missing_train[:12]}")
if missing_val:
    print(f"val perts not in gene_columns: {len(missing_val)}. Example: {missing_val[:12]}")
if missing_sub:
    print(f"sub perts not in gene_columns: {len(missing_sub)}. Example: {missing_sub[:12]}")

if len(missing_all) > 0:
    try:
        print("[h5ad] building embeddings for missing perts:", len(missing_all))
        adata = ad.read_h5ad(str(H5AD_PATH))

        pert_col = None
        for c in ["sgrna_symbol", "pert_symbol", "pert", "perturbation", "gene", "target_gene"]:
            if c in adata.obs.columns:
                pert_col = c
                break
        if pert_col is None:
            raise ValueError("Could not find perturbation column in h5ad obs.")

        Xc = adata.X
        if not sp.issparse(Xc):
            Xc = sp.csr_matrix(Xc)
        else:
            Xc = Xc.tocsr()

        # CPM10K then log2(1+x)
        cell_sum = np.asarray(Xc.sum(axis=1)).ravel().astype(np.float64)
        scale = (10000.0 / cell_sum).astype(np.float64)
        Xn = Xc.multiply(scale[:, None]).tocsr()
        Xn.data = np.log1p(Xn.data) / np.log(2.0)

        ctrl_mask = (adata.obs[pert_col].astype(str).to_numpy() == "non-targeting")
        if int(ctrl_mask.sum()) == 0:
            raise ValueError("No non-targeting control cells found in h5ad.")

        var = {str(g).upper(): i for i, g in enumerate(adata.var_names.astype(str).to_numpy())}

        out_idx = np.array([var[str(g).upper()] for g in gene_columns if str(g).upper() in var], dtype=np.int64)
        if len(out_idx) != len(gene_columns):
            miss_out = [g for g in gene_columns if str(g).upper() not in var]
            raise ValueError(f"{len(miss_out)} output genes missing from h5ad var_names. Example: {miss_out[:10]}")

        Xout = Xn[ctrl_mask][:, out_idx]
        if sp.issparse(Xout):
            Xout = Xout.toarray()
        Xout = Xout.astype(np.float32)

        mu = Xout.mean(axis=0, keepdims=True)
        sd = Xout.std(axis=0, keepdims=True) + 1e-6
        Xout_z = (Xout - mu) / sd

        # use existing base pert space: (G, d_pert)
        P_out = gene_emb_all[:, :EMB_DIM_PERT].astype(np.float32)

        topk = 256
        made = 0
        for gU0 in missing_all:
            if gU0 not in var:
                continue

            xg = Xn[ctrl_mask, var[gU0]]
            if sp.issparse(xg):
                xg = xg.toarray()
            xg = np.asarray(xg).ravel().astype(np.float32)
            xg = (xg - xg.mean()) / (xg.std() + 1e-6)

            corr = (xg[:, None] * Xout_z).mean(axis=0)  # (G,)
            idx = np.argsort(-np.abs(corr))[:topk]
            w = corr[idx].astype(np.float32)

            z = (w[:, None] * P_out[idx]).sum(axis=0)
            z = z / (np.linalg.norm(z) + 1e-12)

            missing_emb_pert[gU0] = z.astype(np.float32)
            made += 1

        print("[h5ad] embedded missing perts:", made, "of", len(missing_all))
    except Exception as e:
        print("[h5ad] skipped missing pert embeddings due to error:", repr(e))

# refresh base Z_train after potential h5ad fills
Z_train_base = np.vstack([emb_pert(g) for g in train_genes]).astype(np.float32)


train perts not in gene_columns: 8. Example: ['BRD4', 'CHD4', 'DNAJA3', 'INO80', 'KAT8', 'KDM4A', 'PMEL', 'SETD1A']
val perts not in gene_columns: 8. Example: ['SMARCB1', 'PSMA1', 'CUL1', 'FLT4', 'FOXH1', 'HK2', 'TRAM2', 'DPH2']
sub perts not in gene_columns: 68. Example: ['SMARCB1', 'PSMA1', 'CUL1', 'FLT4', 'FOXH1', 'HK2', 'TRAM2', 'DPH2', 'pert_61', 'pert_62', 'pert_63', 'pert_64']
[h5ad] building embeddings for missing perts: 76
[h5ad] embedded missing perts: 16 of 76


In [5]:
Y = D_train.astype(np.float32)
G = Y.shape[1]
N = Y.shape[0]

Uo_base_t = torch.tensor(U_out_base, device=device, dtype=torch.float32)  # (G, d_out)
Z_base_t  = torch.tensor(Z_train_base, device=device, dtype=torch.float32)  # (N, d_pert)
Yt = torch.tensor(Y, device=device, dtype=torch.float32)  # (N, G)

# global baseline delta for shrink (broadcast)
delta_baseline_vec = D_train.mean(axis=0).astype(np.float32)          # (G,)
delta_baseline = np.tile(delta_baseline_vec[None, :], (N, 1)).astype(np.float32)

print("N:", N, "G:", G, "device:", device)

N: 80 G: 5127 device: cuda


In [6]:
def gate_smoothstep(x, a=GATE_A, b=GATE_B):
    t = (x - a) / (b - a)
    t = torch.clamp(t, 0.0, 1.0)
    return t * t * (3.0 - 2.0 * t)

def per_row_weighted_l1_like(delta_true: torch.Tensor, delta_pred: torch.Tensor, eps: float = EPS) -> torch.Tensor:
    w = gate_smoothstep(torch.abs(delta_true), a=GATE_A, b=GATE_B)  # (B, G)
    err = torch.abs(delta_pred - delta_true)
    num = torch.sum(w * err, dim=1)
    den = torch.clamp(torch.sum(w, dim=1), min=eps)
    return num / den

def weighted_l1_like_rowweighted(
    delta_true: torch.Tensor,
    delta_pred: torch.Tensor,
    baseline_wmae: torch.Tensor,
    *,
    eps: float = 1e-8,
    mode: str = "inv_sqrt",
    clamp_min: float = 0.5,
    clamp_max: float = 3.0,
) -> torch.Tensor:
    per_row = per_row_weighted_l1_like(delta_true, delta_pred, eps=eps)
    b = baseline_wmae.to(delta_true.device).to(delta_true.dtype)

    if mode == "inv":
        w = 1.0 / (b + eps)
    elif mode == "inv_sqrt":
        w = 1.0 / torch.sqrt(b + eps)
    elif mode == "inv_log":
        w = 1.0 / torch.log1p(b + eps)
    else:
        raise ValueError(f"Unknown mode={mode}")

    w = torch.clamp(w, min=clamp_min, max=clamp_max)
    return torch.sum(w * per_row) / torch.clamp(torch.sum(w), min=eps)


In [7]:
import h5py

STRING_DIR = ROOT / "external" / "string"
ALIASES_PATH = STRING_DIR / "9606.protein.aliases.v12.0.txt"
SEQ_H5_PATH  = STRING_DIR / "9606.protein.sequence.embeddings.v12.0.h5"
NET_H5_PATH  = STRING_DIR / "9606.protein.network.embeddings.v12.0.h5"

for p in [ALIASES_PATH, SEQ_H5_PATH, NET_H5_PATH]:
    print(("OK " if p.exists() else "MISSING "), p)

CACHE_ALIAS = STRING_DIR / "_cache_myllia"
CACHE_ALIAS.mkdir(parents=True, exist_ok=True)

needed_genesU = sorted(set(
    [str(g).upper() for g in gene_columns] +
    [str(g).upper() for g in train_genes.tolist()] +
    [str(g).upper() for g in val_targets] +
    [str(g).upper() for g in sub_perts]
))
print("needed gene symbols:", len(needed_genesU))

MAP_CACHE = CACHE_ALIAS / f"gene2prot_needed_{len(needed_genesU)}.json"

def looks_like_gene_symbol(s: str) -> bool:
    if s is None:
        return False
    if len(s) < 2 or len(s) > 20:
        return False
    if not re.fullmatch(r"[A-Z0-9\.\-]+", s):
        return False
    if re.fullmatch(r"\d+", s):
        return False
    return True

def build_gene2prot_streaming(aliases_path: Path, needed_set: set) -> dict:
    gene2prot = {g: set() for g in needed_set}
    hits = 0
    seen = 0
    with open(aliases_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if not line or line[0] == "#":
                continue
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 3:
                continue
            prot, alias, src = parts[0], parts[1], parts[2]
            aliasU = alias.upper()
            if aliasU in gene2prot and looks_like_gene_symbol(aliasU):
                gene2prot[aliasU].add(prot)
                hits += 1
            seen += 1
            if seen % 5_000_000 == 0:
                print(f"[aliases] lines={seen:,} hits={hits:,}")
    return {g: sorted(list(v)) for g, v in gene2prot.items() if len(v) > 0}

# rescue helpers (your v2)
import re as _re

def _canon_gene(s: str) -> str:
    s = str(s).upper().strip()
    if ":" in s:
        s = s.split(":")[-1]
    s = _re.sub(r"\.\d+$", "", s)
    return s

def _norm_key(s: str) -> str:
    s = _canon_gene(s)
    s = s.replace(" ", "").replace("_", "").replace("-", "")
    return s

def _candidate_keys_for_missing(g: str, try_hyphen_split: bool = False):
    g0 = _canon_gene(g)
    cands = [g0]
    if try_hyphen_split and "-" in g0:
        parts = [p for p in g0.split("-") if p]
        cands += [p for p in parts if looks_like_gene_symbol(p)]
    keys = [_norm_key(x) for x in cands]
    out, seen = [], set()
    for k in keys:
        if k not in seen:
            out.append(k)
            seen.add(k)
    return out

RESCUE_CACHE = CACHE_ALIAS / f"gene2prot_needed_{len(needed_genesU)}_rescue_v2.json"

if RESCUE_CACHE.exists():
    gene2prot = json.loads(RESCUE_CACHE.read_text(encoding="utf-8"))
    print("[cache] loaded rescue gene2prot:", len(gene2prot))
else:
    if MAP_CACHE.exists():
        gene2prot = json.loads(MAP_CACHE.read_text(encoding="utf-8"))
        print("[cache] loaded gene2prot:", len(gene2prot))
    else:
        gene2prot = build_gene2prot_streaming(ALIASES_PATH, set(needed_genesU))
        MAP_CACHE.write_text(json.dumps(gene2prot), encoding="utf-8")
        print("[cache] wrote gene2prot:", len(gene2prot))

    mapped_needed = set(gene2prot.keys())
    missing_needed = [g for g in needed_genesU if g not in mapped_needed]
    print("[rescue] missing before:", len(missing_needed), "example:", missing_needed[:10])

    if len(missing_needed) > 0:
        norm_to_gene = {}
        collisions = set()
        TRY_HYPHEN_SPLIT = False

        for g in missing_needed:
            for k in _candidate_keys_for_missing(g, try_hyphen_split=TRY_HYPHEN_SPLIT):
                if k in norm_to_gene and norm_to_gene[k] != g:
                    collisions.add(k)
                else:
                    norm_to_gene[k] = g
        for k in collisions:
            norm_to_gene.pop(k, None)

        gene2prot2 = {g: set(pids) for g, pids in gene2prot.items()}
        for g in missing_needed:
            gene2prot2.setdefault(g, set())

        hits = 0
        seen = 0
        with open(ALIASES_PATH, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                if not line or line[0] == "#":
                    continue
                parts = line.rstrip("\n").split("\t")
                if len(parts) < 3:
                    continue
                prot, alias, src = parts[0], parts[1], parts[2]
                k = _norm_key(alias)
                g = norm_to_gene.get(k)
                if g is not None:
                    gene2prot2[g].add(prot)
                    hits += 1
                seen += 1
                if seen % 5_000_000 == 0:
                    print(f"[rescue aliases] lines={seen:,} hits={hits:,}")

        gene2prot = {g: sorted(list(v)) for g, v in gene2prot2.items() if len(v) > 0}
        RESCUE_CACHE.write_text(json.dumps(gene2prot), encoding="utf-8")
        print("[cache] wrote rescue gene2prot:", len(gene2prot))

# h5 loader
def decode_obj_bytes(arr):
    if arr is None:
        return None
    out = []
    for x in arr:
        if isinstance(x, (bytes, bytearray)):
            out.append(x.decode("utf-8"))
        else:
            out.append(str(x))
    return np.array(out, dtype=object)

def load_string_h5(path: Path):
    with h5py.File(path, "r") as f:
        ds_names = []
        def walk(name, obj):
            if isinstance(obj, h5py.Dataset):
                ds_names.append(name)
        f.visititems(walk)

        prot_key = None
        for name in ds_names:
            ds = f[name]
            if ds.ndim == 1 and ds.dtype.kind in ("S","O","U"):
                try:
                    v0 = ds[0]
                except Exception:
                    continue
                if isinstance(v0, (bytes, bytearray)) and (b"9606." in v0) and (b"ENSP" in v0):
                    prot_key = name
                    break
        if prot_key is None:
            for name in ds_names:
                ds = f[name]
                if ds.ndim == 1 and ds.dtype.kind in ("S","O","U"):
                    prot_key = name
                    break
        if prot_key is None:
            raise ValueError(f"Could not find protein id dataset in {path}")

        prot = decode_obj_bytes(f[prot_key][:])

        emb_key = None
        for name in ds_names:
            ds = f[name]
            if ds.ndim == 2 and ds.dtype.kind in ("f","i") and ds.shape[0] == len(prot):
                emb_key = name
                break
        if emb_key is None:
            best, best_n = None, -1
            for name in ds_names:
                ds = f[name]
                if ds.ndim == 2 and ds.dtype.kind in ("f","i"):
                    n = ds.shape[0] * ds.shape[1]
                    if n > best_n:
                        best_n = n
                        best = name
            emb_key = best
        if emb_key is None:
            raise ValueError(f"Could not find embedding dataset in {path}")

        emb = np.asarray(f[emb_key][:], dtype=np.float32)

    return prot, emb, {"prot_key": prot_key, "emb_key": emb_key, "shape": emb.shape}

seq_prot, seq_emb_all, _ = load_string_h5(SEQ_H5_PATH)
net_prot, net_emb_all, _ = load_string_h5(NET_H5_PATH)

seq_idx = {pid: i for i, pid in enumerate(seq_prot.tolist())}
net_idx = {pid: i for i, pid in enumerate(net_prot.tolist())}

def mean_emb_for_gene(gU: str, gene2prot: dict, idx_map: dict, emb_all: np.ndarray):
    pids = gene2prot.get(gU)
    if not pids:
        return None
    rows = [idx_map.get(pid) for pid in pids]
    rows = [r for r in rows if r is not None]
    if len(rows) == 0:
        return None
    return emb_all[rows].mean(axis=0).astype(np.float32)

def l2_normalize_vec(v, eps=1e-8):
    n = float(np.linalg.norm(v))
    if n < eps:
        return v
    return (v / (n + eps)).astype(np.float32)

seq_dim = int(seq_emb_all.shape[1])
net_dim = int(net_emb_all.shape[1])

# build output-side seq/net
U_seq = np.zeros((len(gene_columns), seq_dim), dtype=np.float32)
U_net = np.zeros((len(gene_columns), net_dim), dtype=np.float32)
U_seq_mask = np.zeros((len(gene_columns), 1), dtype=np.float32)
U_net_mask = np.zeros((len(gene_columns), 1), dtype=np.float32)

for i, g in enumerate(gene_columns):
    gU0 = str(g).upper()
    vs = mean_emb_for_gene(gU0, gene2prot, seq_idx, seq_emb_all)
    vn = mean_emb_for_gene(gU0, gene2prot, net_idx, net_emb_all)
    if vs is not None:
        U_seq[i] = l2_normalize_vec(vs)
        U_seq_mask[i, 0] = 1.0
    if vn is not None:
        U_net[i] = l2_normalize_vec(vn)
        U_net_mask[i, 0] = 1.0

# build pert-side seq/net (train only)
Z_seq = np.zeros((len(train_genes), seq_dim), dtype=np.float32)
Z_net = np.zeros((len(train_genes), net_dim), dtype=np.float32)
Z_seq_mask = np.zeros((len(train_genes), 1), dtype=np.float32)
Z_net_mask = np.zeros((len(train_genes), 1), dtype=np.float32)

for i, g in enumerate(train_genes.tolist()):
    gU0 = str(g).upper()
    vs = mean_emb_for_gene(gU0, gene2prot, seq_idx, seq_emb_all)
    vn = mean_emb_for_gene(gU0, gene2prot, net_idx, net_emb_all)
    if vs is not None:
        Z_seq[i] = l2_normalize_vec(vs)
        Z_seq_mask[i, 0] = 1.0
    if vn is not None:
        Z_net[i] = l2_normalize_vec(vn)
        Z_net_mask[i, 0] = 1.0

print("[seq/net] U_seq cov:", float(U_seq_mask.mean()), "U_net cov:", float(U_net_mask.mean()),
      "Z_seq cov:", float(Z_seq_mask.mean()), "Z_net cov:", float(Z_net_mask.mean()))

OK  external\string\9606.protein.aliases.v12.0.txt
OK  external\string\9606.protein.sequence.embeddings.v12.0.h5
OK  external\string\9606.protein.network.embeddings.v12.0.h5
needed gene symbols: 5203
[cache] loaded rescue gene2prot: 5056
[seq/net] U_seq cov: 0.9830310344696045 U_net cov: 0.9830310344696045 Z_seq cov: 0.987500011920929 Z_net cov: 0.987500011920929


In [8]:
U_seq_t = torch.tensor(U_seq, device=device, dtype=torch.float32)
U_net_t = torch.tensor(U_net, device=device, dtype=torch.float32)
Z_seq_t = torch.tensor(Z_seq, device=device, dtype=torch.float32)
Z_net_t = torch.tensor(Z_net, device=device, dtype=torch.float32)

U_seq_mask_t = torch.tensor(U_seq_mask, device=device, dtype=torch.float32)
U_net_mask_t = torch.tensor(U_net_mask, device=device, dtype=torch.float32)
Z_seq_mask_t = torch.tensor(Z_seq_mask, device=device, dtype=torch.float32)
Z_net_mask_t = torch.tensor(Z_net_mask, device=device, dtype=torch.float32)

In [9]:
def find_file(filename: str) -> Path:
    candidates = [
        ROOT / "data" / filename,
        ROOT / "Data" / filename,
        ROOT / "external" / filename,
        ROOT / "external" / "STRING" / filename,
        ROOT / "external" / "string" / filename,
        ROOT / "string" / filename,
        ROOT / filename,
    ]
    for p in candidates:
        if p.exists():
            return p
    for p in ROOT.rglob(filename):
        return p
    raise FileNotFoundError(f"Could not find {filename} under {ROOT.resolve()}")

def load_ensp_to_symbol_from_protein_info(info_path: Path):
    df = pd.read_csv(info_path, sep="\t", dtype=str)
    if df.shape[1] < 2:
        df = pd.read_csv(info_path, sep=r"\s+", dtype=str)
    cols = list(df.columns)
    if cols and isinstance(cols[0], str) and cols[0].startswith("#"):
        df = df.rename(columns={cols[0]: cols[0].lstrip("#")})
    id_col = df.columns[0]
    name_col = df.columns[1]
    prot = df[id_col].astype(str).to_numpy()
    name = df[name_col].astype(str).to_numpy()
    prot = np.char.replace(prot.astype("U"), "9606.", "")
    name = np.char.upper(name.astype("U"))
    return dict(zip(prot.tolist(), name.tolist()))

def l2norm_rows(X: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    X = X.astype(np.float32, copy=False)
    n = np.linalg.norm(X, axis=1, keepdims=True)
    return X / (n + eps)

def csr_topk(A_csr: sp.csr_matrix, k: int):
    A = A_csr.tocsr()
    A.sort_indices()
    indptr, indices, data = A.indptr, A.indices, A.data
    rr, cc, dd = [], [], []
    for i in range(A.shape[0]):
        s, e = indptr[i], indptr[i + 1]
        if e <= s:
            continue
        row_idx = indices[s:e]
        row_dat = data[s:e]
        if (e - s) > k:
            sel = np.argpartition(row_dat, -k)[-k:]
            row_idx = row_idx[sel]
            row_dat = row_dat[sel]
        rr.append(np.full(len(row_idx), i, dtype=np.int32))
        cc.append(row_idx.astype(np.int32))
        dd.append(row_dat.astype(np.float32))
    if not rr:
        return sp.csr_matrix(A.shape, dtype=np.float32)
    r = np.concatenate(rr)
    c = np.concatenate(cc)
    d = np.concatenate(dd)
    return sp.coo_matrix((d, (r, c)), shape=A.shape).tocsr()

def build_adj_from_protein_links(
    links_path: Path,
    ensp2gi: dict,
    n_genes: int,
    score_min: int = 700,
    topk: int = 200,
    edge_gamma: float = 1.0,
    chunksize: int = 2_000_000,
    cache_npz=None,
):
    if cache_npz is not None and cache_npz.exists():
        print("[cache] load adj:", cache_npz)
        return sp.load_npz(cache_npz).tocsr()

    usecols = ["protein1", "protein2", "combined_score"]
    dtypes = {"protein1": "string", "protein2": "string", "combined_score": np.int32}

    rows_parts, cols_parts, dat_parts = [], [], []
    kept_edges = 0

    it = pd.read_csv(links_path, sep=r"\s+", usecols=usecols, dtype=dtypes, chunksize=chunksize)
    for ci, chunk in enumerate(it, 1):
        s = chunk["combined_score"].to_numpy()
        m_score = s >= int(score_min)
        if not m_score.any():
            continue

        p1 = chunk["protein1"].to_numpy()[m_score]
        p2 = chunk["protein2"].to_numpy()[m_score]
        w  = (s[m_score].astype(np.float32) / 1000.0)
        if float(edge_gamma) != 1.0:
            w = np.power(w, float(edge_gamma))

        p1 = np.char.replace(p1.astype("U"), "9606.", "")
        p2 = np.char.replace(p2.astype("U"), "9606.", "")

        i = np.fromiter((ensp2gi.get(x, -1) for x in p1), dtype=np.int32, count=len(p1))
        j = np.fromiter((ensp2gi.get(x, -1) for x in p2), dtype=np.int32, count=len(p2))

        m = (i >= 0) & (j >= 0) & (i != j)
        if not m.any():
            continue

        ii = i[m]
        jj = j[m]
        ww = w[m]

        rows_parts.append(ii)
        cols_parts.append(jj)
        dat_parts.append(ww)
        kept_edges += len(ii)

        if ci % 5 == 0:
            print(f"[parse] chunks={ci} kept_edges={kept_edges:,}")

    if not rows_parts:
        A = sp.csr_matrix((n_genes, n_genes), dtype=np.float32)
    else:
        r = np.concatenate(rows_parts)
        c = np.concatenate(cols_parts)
        d = np.concatenate(dat_parts)
        A = sp.coo_matrix((d, (r, c)), shape=(n_genes, n_genes), dtype=np.float32).tocsr()

    A = A + A.T
    A.sum_duplicates()

    if topk is not None and int(topk) > 0:
        A = csr_topk(A, int(topk))
        A = A + A.T
        A.sum_duplicates()

    A = sk_normalize(A, norm="l1", axis=1)

    if cache_npz is not None:
        cache_npz.parent.mkdir(parents=True, exist_ok=True)
        print("[cache] save adj:", cache_npz)
        sp.save_npz(cache_npz, A)

    return A

def embed_from_adj(A_csr: sp.csr_matrix, emb_dim: int = 128, seed: int = 6):
    svd = TruncatedSVD(n_components=int(emb_dim), random_state=int(seed))
    U = svd.fit_transform(A_csr).astype(np.float32)
    U = sk_normalize(U, norm="l2", axis=1)
    return U

def embed_from_adj_multihop(
    A_csr: sp.csr_matrix,
    emb_dim: int = 128,
    beta2: float = 1.0,
    beta3: float = 0.0,
    seed: int = 6,
    hop_topk: int = 400,
):
    A1 = sk_normalize(A_csr, norm="l1", axis=1)

    A2 = (A1 @ A1).tocsr()
    A2.sum_duplicates()
    if hop_topk is not None and int(hop_topk) > 0:
        A2 = csr_topk(A2, int(hop_topk))
    A2 = sk_normalize(A2, norm="l1", axis=1)

    Amix = (A1 + float(beta2) * A2).tocsr()
    Amix.sum_duplicates()

    if float(beta3) != 0.0:
        A3 = (A2 @ A1).tocsr()
        A3.sum_duplicates()
        if hop_topk is not None and int(hop_topk) > 0:
            A3 = csr_topk(A3, int(hop_topk))
        A3 = sk_normalize(A3, norm="l1", axis=1)

        Amix = (Amix + float(beta3) * A3).tocsr()
        Amix.sum_duplicates()

    return embed_from_adj(Amix, emb_dim=int(emb_dim), seed=int(seed))

def build_string_links_universe_embeddings(cfg: dict, emb_dim: int, seed: int):
    links_path = find_file("9606.protein.links.v12.0.txt")
    info_path  = find_file("9606.protein.info.v12.0.txt")

    # include submission perts too (so inference never sees "unknown" symbols)
    universe = sorted(set(
        [str(g).upper() for g in gene_columns] +
        [str(g).upper() for g in train_genes.tolist()] +
        [str(g).upper() for g in val_targets] +
        [str(g).upper() for g in sub_perts]
    ))
    g2i = {g: i for i, g in enumerate(universe)}

    i_out  = np.array([g2i[str(g).upper()] for g in gene_columns], dtype=np.int32)
    i_pert = np.array([g2i.get(str(g).upper(), -1) for g in train_genes.tolist()], dtype=np.int32)

    ensp2sym = load_ensp_to_symbol_from_protein_info(info_path)

    # map ENSP -> gene index in our universe
    ensp2gi = {}
    mapped_flag = np.zeros((len(universe),), dtype=np.float32)
    for ensp, sym in ensp2sym.items():
        gi = g2i.get(sym, None)
        if gi is not None:
            ensp2gi[ensp] = gi
            mapped_flag[gi] = 1.0

    cache_npz = CACHE_LINKS / f"adj_{cfg['key']}_n{len(universe)}.npz"
    A = build_adj_from_protein_links(
        links_path=links_path,
        ensp2gi=ensp2gi,
        n_genes=len(universe),
        score_min=int(cfg["score_min"]),
        topk=int(cfg["topk"]),
        edge_gamma=float(cfg["edge_gamma"]),
        chunksize=2_000_000,
        cache_npz=cache_npz,
    )

    # degree mask (after pruning/normalization)
    deg = np.asarray(A.sum(axis=1)).ravel().astype(np.float32)
    deg_mask = (deg > 0).astype(np.float32)[:, None]

    hop_topk = int(cfg["topk"])
    cache_emb = CACHE_LINKS / f"U_{cfg['key']}_k{emb_dim}_n{len(universe)}.npy"
    if cache_emb.exists():
        print("[cache] load links emb:", cache_emb)
        U = np.load(cache_emb).astype(np.float32)
    else:
        U = embed_from_adj_multihop(
            A,
            emb_dim=int(emb_dim),
            beta2=float(cfg["beta2"]),
            beta3=float(cfg["beta3"]),
            seed=int(seed),
            hop_topk=hop_topk,
        ).astype(np.float32)
        cache_emb.parent.mkdir(parents=True, exist_ok=True)
        print("[cache] save links emb:", cache_emb)
        np.save(cache_emb, U)

    u_mean = U.mean(axis=0).astype(np.float32)

    # aligned slices
    U_out_link = U[i_out].astype(np.float32)
    U_out_link_mask = deg_mask[i_out].astype(np.float32)

    Z_train_link = np.vstack([(U[i] if i >= 0 else u_mean) for i in i_pert]).astype(np.float32)
    Z_train_link_mask = np.vstack([(deg_mask[i] if i >= 0 else np.zeros((1,), np.float32)) for i in i_pert]).astype(np.float32)

    return {
        "universe": universe,
        "g2i": g2i,
        "U": U,
        "u_mean": u_mean,
        "deg_mask": deg_mask,
        "U_out_link": U_out_link,
        "U_out_link_mask": U_out_link_mask,
        "Z_train_link": Z_train_link,
        "Z_train_link_mask": Z_train_link_mask,
    }

links_pack = build_string_links_universe_embeddings(CFG_B, emb_dim=EMB_DIM_OUT, seed=SEED)

U_link = links_pack["U_out_link"]
U_link_mask = links_pack["U_out_link_mask"]
Z_link = links_pack["Z_train_link"]
Z_link_mask = links_pack["Z_train_link_mask"]

print("[links] U_link cov:", float(U_link_mask.mean()), "Z_link cov:", float(Z_link_mask.mean()))

U_link_t = torch.tensor(U_link, device=device, dtype=torch.float32)
Z_link_t = torch.tensor(Z_link, device=device, dtype=torch.float32)
U_link_mask_t = torch.tensor(U_link_mask, device=device, dtype=torch.float32)
Z_link_mask_t = torch.tensor(Z_link_mask, device=device, dtype=torch.float32)

[cache] load adj: cache\string_links_v12\adj_B_rank1_eg2_b2=1_b3=0.5_n5203.npz
[cache] load links emb: cache\string_links_v12\U_B_rank1_eg2_b2=1_b3=0.5_k128_n5203.npy
[links] U_link cov: 0.9602106213569641 Z_link cov: 0.987500011920929


In [10]:
# Final model: 4-block gated bilinear
#   base + seq + net + links
# -----------------------
class HybridAllStringBilinearDeltaModel(nn.Module):
    def __init__(self, d_base_p, d_base_o, d_seq, d_net, d_link, rank_r, dropout):
        super().__init__()
        self.rank_r = int(rank_r)

        # pert towers
        self.p_base = nn.Sequential(nn.Linear(d_base_p, rank_r), nn.GELU(), nn.Dropout(dropout))
        self.p_seq  = nn.Sequential(nn.Linear(d_seq,   rank_r, bias=False), nn.GELU(), nn.Dropout(dropout))
        self.p_net  = nn.Sequential(nn.Linear(d_net,   rank_r, bias=False), nn.GELU(), nn.Dropout(dropout))
        self.p_link = nn.Sequential(nn.Linear(d_link,  rank_r, bias=False), nn.GELU(), nn.Dropout(dropout))

        # out towers
        self.o_base = nn.Sequential(nn.Linear(d_base_o, rank_r), nn.GELU(), nn.Dropout(dropout))
        self.o_seq  = nn.Sequential(nn.Linear(d_seq,    rank_r, bias=False), nn.GELU(), nn.Dropout(dropout))
        self.o_net  = nn.Sequential(nn.Linear(d_net,    rank_r, bias=False), nn.GELU(), nn.Dropout(dropout))
        self.o_link = nn.Sequential(nn.Linear(d_link,   rank_r, bias=False), nn.GELU(), nn.Dropout(dropout))

        # gating
        self.gate_p = nn.Sequential(
            nn.Linear(4 * rank_r, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 4),
        )
        self.gate_o = nn.Sequential(
            nn.Linear(4 * rank_r, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 4),
        )

        self.bias_global = nn.Parameter(torch.zeros(1))
        self.bias_gene = None

    def set_gene_bias(self, G):
        dev = next(self.parameters()).device
        self.bias_gene = nn.Parameter(torch.zeros(G, device=dev))
        self.bias_global = nn.Parameter(torch.zeros(1, device=dev))

    @staticmethod
    def _mask_logits_4(logits, m_seq, m_net, m_link, big=1e9):
        # logits: (B,4), masks: (B,1) float 0/1
        out = logits
        if m_seq is not None:
            out = out.clone()
            out[:, 1] = out[:, 1] + (m_seq.squeeze(1) - 1.0) * big
        if m_net is not None:
            out = out.clone()
            out[:, 2] = out[:, 2] + (m_net.squeeze(1) - 1.0) * big
        if m_link is not None:
            out = out.clone()
            out[:, 3] = out[:, 3] + (m_link.squeeze(1) - 1.0) * big
        return out

    def forward(
        self,
        z_base, z_seq, z_net, z_link,
        u_base, u_seq, u_net, u_link,
        z_seq_mask=None, z_net_mask=None, z_link_mask=None,
        u_seq_mask=None, u_net_mask=None, u_link_mask=None,
    ):
        pb = self.p_base(z_base)
        ps = self.p_seq(z_seq)
        pn = self.p_net(z_net)
        pl = self.p_link(z_link)

        ob = self.o_base(u_base)
        os = self.o_seq(u_seq)
        on = self.o_net(u_net)
        ol = self.o_link(u_link)

        # default masks
        if z_seq_mask is None:
            z_seq_mask = torch.ones((z_base.shape[0], 1), device=z_base.device, dtype=z_base.dtype)
        if z_net_mask is None:
            z_net_mask = torch.ones((z_base.shape[0], 1), device=z_base.device, dtype=z_base.dtype)
        if z_link_mask is None:
            z_link_mask = torch.ones((z_base.shape[0], 1), device=z_base.device, dtype=z_base.dtype)

        if u_seq_mask is None:
            u_seq_mask = torch.ones((u_base.shape[0], 1), device=u_base.device, dtype=u_base.dtype)
        if u_net_mask is None:
            u_net_mask = torch.ones((u_base.shape[0], 1), device=u_base.device, dtype=u_base.dtype)
        if u_link_mask is None:
            u_link_mask = torch.ones((u_base.shape[0], 1), device=u_base.device, dtype=u_base.dtype)

        gp_logits = self.gate_p(torch.cat([pb, ps, pn, pl], dim=1))
        gp_logits = self._mask_logits_4(gp_logits, z_seq_mask, z_net_mask, z_link_mask)
        gp = torch.softmax(gp_logits, dim=1)

        go_logits = self.gate_o(torch.cat([ob, os, on, ol], dim=1))
        go_logits = self._mask_logits_4(go_logits, u_seq_mask, u_net_mask, u_link_mask)
        go = torch.softmax(go_logits, dim=1)

        p = gp[:, 0:1] * pb + gp[:, 1:2] * ps + gp[:, 2:3] * pn + gp[:, 3:4] * pl
        o = go[:, 0:1] * ob + go[:, 1:2] * os + go[:, 2:3] * on + go[:, 3:4] * ol

        y = p @ o.T
        y = y + self.bias_gene[None, :] + self.bias_global
        return y

In [11]:
def train_one_fold_all(tr_idx, va_idx, seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = HybridAllStringBilinearDeltaModel(
        d_base_p=Z_base_t.shape[1],
        d_base_o=Uo_base_t.shape[1],
        d_seq=U_seq_t.shape[1],
        d_net=U_net_t.shape[1],
        d_link=U_link_t.shape[1],
        rank_r=RANK_R,
        dropout=DROPOUT,
    ).to(device)

    model.set_gene_bias(G)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    tr_idx = np.asarray(tr_idx)
    va_idx = np.asarray(va_idx)

    va_idx_t = torch.tensor(va_idx, device=device, dtype=torch.long)
    va_true = Y[va_idx]                   # (B,G) numpy
    va_base = delta_baseline[va_idx]      # (B,G) numpy

    best_score = -1e18
    best_alpha = 0.0
    best_epoch = 0
    best_state = None
    best_va_raw = None
    patience = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = tr_idx.copy()
        np.random.shuffle(perm)

        for start in range(0, len(perm), BATCH_GENES):
            b = perm[start:start + BATCH_GENES]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(
                Z_base_t.index_select(0, b_t),
                Z_seq_t.index_select(0, b_t),
                Z_net_t.index_select(0, b_t),
                Z_link_t.index_select(0, b_t),
                Uo_base_t,
                U_seq_t,
                U_net_t,
                U_link_t,
                Z_seq_mask_t.index_select(0, b_t),
                Z_net_mask_t.index_select(0, b_t),
                Z_link_mask_t.index_select(0, b_t),
                U_seq_mask_t,
                U_net_mask_t,
                U_link_mask_t,
            )

            dt_b = Yt.index_select(0, b_t)
            bw_b = baseline_wmae_t.index_select(0, b_t)

            loss = weighted_l1_like_rowweighted(
                dt_b, pred, bw_b,
                mode="inv_sqrt",
                clamp_min=0.5,
                clamp_max=3.0,
            )

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

        if (epoch % EVAL_EVERY == 0) or (epoch == EPOCHS):
            model.eval()
            with torch.no_grad():
                va_pred = model(
                    Z_base_t.index_select(0, va_idx_t),
                    Z_seq_t.index_select(0, va_idx_t),
                    Z_net_t.index_select(0, va_idx_t),
                    Z_link_t.index_select(0, va_idx_t),
                    Uo_base_t,
                    U_seq_t,
                    U_net_t,
                    U_link_t,
                    Z_seq_mask_t.index_select(0, va_idx_t),
                    Z_net_mask_t.index_select(0, va_idx_t),
                    Z_link_mask_t.index_select(0, va_idx_t),
                    U_seq_mask_t,
                    U_net_mask_t,
                    U_link_mask_t,
                ).detach().cpu().numpy().astype(np.float32)

            # tune alpha on val
            sc_best = -1e18
            a_best = 0.0
            for a in ALPHA_GRID:
                pred_s = apply_shrink(va_pred, va_base, float(a))
                sc = score_delta(va_true, pred_s)["score"]
                if sc > sc_best:
                    sc_best = sc
                    a_best = float(a)

            va_s = apply_shrink(va_pred, va_base, a_best)
            va_score = score_delta(va_true, va_s)["score"]

            if va_score > best_score:
                best_score = float(va_score)
                best_alpha = float(a_best)
                best_epoch = int(epoch)
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                best_va_raw = va_pred
                patience = 0
            else:
                patience += 1

            if patience >= PATIENCE:
                break

    return best_score, best_alpha, best_epoch, best_state, best_va_raw

# %%
kf = KFold(n_splits=8, shuffle=True, random_state=SEED)

oof_raw = np.zeros_like(Y, dtype=np.float32)
oof_hit = np.zeros((N,), dtype=np.int32)

fold_scores = []
fold_alphas = []
fold_epochs = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(np.arange(N)), 1):
    best_score, best_alpha, best_epoch, best_state, best_va_raw = train_one_fold_all(tr_idx, va_idx, seed=SEED)
    fold_scores.append(float(best_score))
    fold_alphas.append(float(best_alpha))
    fold_epochs.append(int(best_epoch))

    oof_raw[va_idx] = best_va_raw
    oof_hit[va_idx] += 1

    print(f"[ALL] fold {fold}: best_score={best_score:.6f} alpha={best_alpha:.3f} epoch={best_epoch}")

if not np.all(oof_hit == 1):
    print("[warn] OOF coverage not 1 everywhere. min/max:", int(oof_hit.min()), int(oof_hit.max()))

print("[ALL] cv mean:", float(np.mean(fold_scores)), "std:", float(np.std(fold_scores)))
EPOCHS_MED = int(np.median(fold_epochs))
print("[ALL] median best_epoch =", EPOCHS_MED)

# global alpha from OOF
best_global_alpha = 0.0
best_global_score = -1e18
for a in ALPHA_GRID:
    pred_a = apply_shrink(oof_raw, delta_baseline, float(a))
    sc = score_delta(Y, pred_a)["score"]
    if sc > best_global_score:
        best_global_score = sc
        best_global_alpha = float(a)

print("[ALL] OOF global alpha:", best_global_alpha, "OOF score:", float(best_global_score))
ALPHA_SHRINK_ALL = float(best_global_alpha)

[ALL] fold 1: best_score=0.163046 alpha=0.860 epoch=30
[ALL] fold 2: best_score=0.142535 alpha=0.900 epoch=160
[ALL] fold 3: best_score=0.102276 alpha=0.520 epoch=25
[ALL] fold 4: best_score=0.188144 alpha=0.900 epoch=100
[ALL] fold 5: best_score=0.192388 alpha=0.820 epoch=155
[ALL] fold 6: best_score=0.191571 alpha=0.720 epoch=65
[ALL] fold 7: best_score=0.162106 alpha=0.720 epoch=15
[ALL] fold 8: best_score=0.112156 alpha=0.640 epoch=15
[ALL] cv mean: 0.15677790495764032 std: 0.03296695599257322
[ALL] median best_epoch = 47
[ALL] OOF global alpha: 0.7799999713897705 OOF score: 0.15360504231628092


[ALL] fold 1: best_score=0.163046 alpha=0.860 epoch=30
[ALL] fold 2: best_score=0.142535 alpha=0.900 epoch=160
[ALL] fold 3: best_score=0.102276 alpha=0.520 epoch=25
[ALL] fold 4: best_score=0.188144 alpha=0.900 epoch=100
[ALL] fold 5: best_score=0.192388 alpha=0.820 epoch=155
[ALL] fold 6: best_score=0.191571 alpha=0.720 epoch=65
[ALL] fold 7: best_score=0.162106 alpha=0.720 epoch=15
[ALL] fold 8: best_score=0.112156 alpha=0.640 epoch=15
[ALL] cv mean: 0.15677790495764032 std: 0.03296695599257322
[ALL] median best_epoch = 47
[ALL] OOF global alpha: 0.7799999713897705 OOF score: 0.15360504231628092

In [12]:
def build_Z_base(perts):
    return np.vstack([emb_pert(p) for p in perts]).astype(np.float32)

def build_Z_seqnet(perts):
    Zs = np.zeros((len(perts), seq_dim), dtype=np.float32)
    Zn = np.zeros((len(perts), net_dim), dtype=np.float32)
    Ms = np.zeros((len(perts), 1), dtype=np.float32)
    Mn = np.zeros((len(perts), 1), dtype=np.float32)

    for i, p in enumerate(perts):
        gU0 = str(p).upper()
        vs = mean_emb_for_gene(gU0, gene2prot, seq_idx, seq_emb_all)
        vn = mean_emb_for_gene(gU0, gene2prot, net_idx, net_emb_all)
        if vs is not None:
            Zs[i] = l2_normalize_vec(vs)
            Ms[i, 0] = 1.0
        if vn is not None:
            Zn[i] = l2_normalize_vec(vn)
            Mn[i, 0] = 1.0
    return Zs, Zn, Ms, Mn

def build_Z_links(perts, links_pack):
    g2i = links_pack["g2i"]
    U = links_pack["U"]
    u_mean = links_pack["u_mean"]
    deg_mask = links_pack["deg_mask"]

    Zl = np.zeros((len(perts), U.shape[1]), dtype=np.float32)
    Ml = np.zeros((len(perts), 1), dtype=np.float32)

    for i, p in enumerate(perts):
        gU0 = str(p).upper()
        idx = g2i.get(gU0, None)
        if idx is None:
            Zl[i] = u_mean
            Ml[i, 0] = 0.0
        else:
            Zl[i] = U[idx]
            Ml[i, 0] = float(deg_mask[idx, 0])
    return Zl, Ml

Z_base_sub = build_Z_base(sub_perts)
Z_seq_sub, Z_net_sub, Z_seq_sub_mask, Z_net_sub_mask = build_Z_seqnet(sub_perts)
Z_link_sub, Z_link_sub_mask = build_Z_links(sub_perts, links_pack)

Z_base_sub_t = torch.tensor(Z_base_sub, device=device, dtype=torch.float32)
Z_seq_sub_t  = torch.tensor(Z_seq_sub,  device=device, dtype=torch.float32)
Z_net_sub_t  = torch.tensor(Z_net_sub,  device=device, dtype=torch.float32)
Z_link_sub_t = torch.tensor(Z_link_sub, device=device, dtype=torch.float32)

Z_seq_sub_mask_t  = torch.tensor(Z_seq_sub_mask,  device=device, dtype=torch.float32)
Z_net_sub_mask_t  = torch.tensor(Z_net_sub_mask,  device=device, dtype=torch.float32)
Z_link_sub_mask_t = torch.tensor(Z_link_sub_mask, device=device, dtype=torch.float32)

# %%
# -----------------------
# Refit on full training set and predict submission
# -----------------------
def fit_full_model_all(seed: int, epochs_fixed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = HybridAllStringBilinearDeltaModel(
        d_base_p=Z_base_t.shape[1],
        d_base_o=Uo_base_t.shape[1],
        d_seq=U_seq_t.shape[1],
        d_net=U_net_t.shape[1],
        d_link=U_link_t.shape[1],
        rank_r=RANK_R,
        dropout=DROPOUT,
    ).to(device)

    model.set_gene_bias(G)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    idx = np.arange(N)
    for epoch in range(1, int(epochs_fixed) + 1):
        model.train()
        np.random.shuffle(idx)

        for start in range(0, N, BATCH_GENES):
            b = idx[start:start + BATCH_GENES]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(
                Z_base_t.index_select(0, b_t),
                Z_seq_t.index_select(0, b_t),
                Z_net_t.index_select(0, b_t),
                Z_link_t.index_select(0, b_t),
                Uo_base_t,
                U_seq_t,
                U_net_t,
                U_link_t,
                Z_seq_mask_t.index_select(0, b_t),
                Z_net_mask_t.index_select(0, b_t),
                Z_link_mask_t.index_select(0, b_t),
                U_seq_mask_t,
                U_net_mask_t,
                U_link_mask_t,
            )

            dt_b = Yt.index_select(0, b_t)
            bw_b = baseline_wmae_t.index_select(0, b_t)

            loss = weighted_l1_like_rowweighted(
                dt_b, pred, bw_b,
                mode="inv_sqrt",
                clamp_min=0.5,
                clamp_max=3.0,
            )

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

        if epoch % 10 == 0 or epoch == epochs_fixed:
            model.eval()
            with torch.no_grad():
                tr_pred = model(
                    Z_base_t, Z_seq_t, Z_net_t, Z_link_t,
                    Uo_base_t, U_seq_t, U_net_t, U_link_t,
                    Z_seq_mask_t, Z_net_mask_t, Z_link_mask_t,
                    U_seq_mask_t, U_net_mask_t, U_link_mask_t,
                ).detach().cpu().numpy().astype(np.float32)
                tr_pred_s = apply_shrink(tr_pred, delta_baseline, ALPHA_SHRINK_ALL)
                tr_score = score_delta(Y, tr_pred_s)["score"]
            print(f"[REFIT][TRAIN] seed={seed} epoch={epoch:4d} train_score={tr_score:.6f} loss={float(loss):.6f}")

    return model

epochs_fixed = int(EPOCHS_MED)
alpha_use = float(ALPHA_SHRINK_ALL)
print("[SUBMIT] epochs_fixed=", epochs_fixed, "alpha=", alpha_use, "seeds=", MODEL_SEEDS, "CFG_LINKS=", CFG_B["key"])

pred_list = []
for s in MODEL_SEEDS:
    m = fit_full_model_all(seed=int(s), epochs_fixed=epochs_fixed)
    m.eval()
    with torch.no_grad():
        p = m(
            Z_base_sub_t, Z_seq_sub_t, Z_net_sub_t, Z_link_sub_t,
            Uo_base_t, U_seq_t, U_net_t, U_link_t,
            Z_seq_sub_mask_t, Z_net_sub_mask_t, Z_link_sub_mask_t,
            U_seq_mask_t, U_net_mask_t, U_link_mask_t,
        ).detach().cpu().numpy().astype(np.float32)
    pred_list.append(p)

pred_raw = np.mean(np.stack(pred_list, axis=0), axis=0).astype(np.float32)  # (rows, G)

# submission baseline: broadcast global mean delta
sub_baseline = np.tile(delta_baseline_vec[None, :], (pred_raw.shape[0], 1)).astype(np.float32)
pred = apply_shrink(pred_raw, sub_baseline, alpha_use).astype(np.float32)

sub_out = df_sub.copy()
sub_out[gene_columns] = pred.astype(np.float64)

out_path = f"submission_hybrid_all_links_{CFG_B['key']}_e{epochs_fixed}_a{alpha_use:.3f}.csv"
sub_out.to_csv(out_path, index=False, float_format="%.20f")
print("[SUBMIT] wrote:", out_path, "shape:", sub_out.shape)

[SUBMIT] epochs_fixed= 47 alpha= 0.7799999713897705 seeds= [6, 7, 8] CFG_LINKS= B_rank1_eg2_b2=1_b3=0.5
[REFIT][TRAIN] seed=6 epoch=  10 train_score=0.173244 loss=0.070296
[REFIT][TRAIN] seed=6 epoch=  20 train_score=0.235768 loss=0.069760
[REFIT][TRAIN] seed=6 epoch=  30 train_score=0.269832 loss=0.061280
[REFIT][TRAIN] seed=6 epoch=  40 train_score=0.297568 loss=0.061920
[REFIT][TRAIN] seed=6 epoch=  47 train_score=0.320746 loss=0.057502
[REFIT][TRAIN] seed=7 epoch=  10 train_score=0.172275 loss=0.066551
[REFIT][TRAIN] seed=7 epoch=  20 train_score=0.231548 loss=0.069863
[REFIT][TRAIN] seed=7 epoch=  30 train_score=0.265929 loss=0.064769
[REFIT][TRAIN] seed=7 epoch=  40 train_score=0.292519 loss=0.064889
[REFIT][TRAIN] seed=7 epoch=  47 train_score=0.316403 loss=0.059943
[REFIT][TRAIN] seed=8 epoch=  10 train_score=0.173323 loss=0.062495
[REFIT][TRAIN] seed=8 epoch=  20 train_score=0.235521 loss=0.074309
[REFIT][TRAIN] seed=8 epoch=  30 train_score=0.267777 loss=0.063350
[REFIT][TRAI